# 01 - Explore Kaggle BSL Data

This notebook explores the Kaggle BSL dataset structure, visualizes landmark distributions,
and identifies data quality issues and preprocessing needs.

## Contents
1. Load and examine dataset structure
2. Visualize landmark distributions
3. Check for data quality issues
4. Compare different sign samples
5. Identify preprocessing needs

In [ ]:
# Common imports
import sys
sys.path.insert(0, '..')

from src.types import *
from src.landmarks import *
from src.validation import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import json
from pathlib import Path
from collections import defaultdict

# Local visualization utilities
from notebook_utils import (
    plot_hand_3d,
    plot_hand_2d,
    plot_hand_views,
    plot_comparison,
    plot_multiple_hands,
    plot_landmark_distributions,
    plot_variance_heatmap,
    landmarks_to_arrays,
    set_notebook_style,
)

set_notebook_style()
%matplotlib inline

print("Imports loaded successfully!")

## 1. Load and Examine Dataset Structure

First, let's explore what files are available and their structure.

In [ ]:
# Configuration
KAGGLE_DIR = Path('../data/kaggle-bsl')

# List available files
print(f"Kaggle data directory: {KAGGLE_DIR}")
print(f"Directory exists: {KAGGLE_DIR.exists()}")
print()

if KAGGLE_DIR.exists():
    files = list(KAGGLE_DIR.iterdir())
    print(f"Found {len(files)} files:")
    for f in sorted(files):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name}: {size_mb:.2f} MB")
else:
    print("Directory not found. Please check the path.")

In [ ]:
# Load CSV file(s)
csv_files = list(KAGGLE_DIR.glob('*.csv'))

if csv_files:
    print(f"Found {len(csv_files)} CSV files")
    
    # Load first CSV to examine structure
    df = pd.read_csv(csv_files[0], nrows=1000)  # Load first 1000 rows for exploration
    
    print(f"\nDataset shape: {df.shape}")
    print(f"Columns: {len(df.columns)}")
    print(f"\nColumn names (first 20):")
    print(df.columns[:20].tolist())
    print(f"\nColumn names (last 20):")
    print(df.columns[-20:].tolist())
else:
    print("No CSV files found")

In [ ]:
# Examine data types and basic stats
if 'df' in dir():
    print("Data types:")
    print(df.dtypes.value_counts())
    print("\nFirst few rows:")
    display(df.head())
    
    print("\nBasic statistics:")
    display(df.describe())

In [ ]:
# Identify landmark columns
# Kaggle BSL dataset typically has columns like: x_right_hand_0, y_right_hand_0, z_right_hand_0, etc.

if 'df' in dir():
    # Look for hand landmark patterns
    right_hand_cols = [c for c in df.columns if 'right_hand' in c.lower()]
    left_hand_cols = [c for c in df.columns if 'left_hand' in c.lower()]
    
    print(f"Right hand columns: {len(right_hand_cols)}")
    print(f"Left hand columns: {len(left_hand_cols)}")
    
    if right_hand_cols:
        print(f"\nSample right hand columns:")
        print(right_hand_cols[:10])

In [ ]:
# Check for sign labels/identifiers
if 'df' in dir():
    # Common label column names
    label_candidates = ['sign', 'label', 'sign_id', 'class', 'target', 'sequence_id']
    
    found_labels = [c for c in df.columns if any(lc in c.lower() for lc in label_candidates)]
    print(f"Potential label columns: {found_labels}")
    
    for col in found_labels:
        print(f"\n{col}:")
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Sample values: {df[col].unique()[:10].tolist()}")

## 2. Visualize Landmark Distributions

In [ ]:
def extract_landmarks_from_row(row, hand='right'):
    """
    Extract landmarks from a DataFrame row.
    Adjust column patterns based on actual dataset structure.
    """
    landmarks = []
    
    for i in range(21):  # 21 landmarks per hand
        # Try different column naming patterns
        patterns = [
            (f'x_{hand}_hand_{i}', f'y_{hand}_hand_{i}', f'z_{hand}_hand_{i}'),
            (f'{hand}_hand_{i}_x', f'{hand}_hand_{i}_y', f'{hand}_hand_{i}_z'),
            (f'x_hand_{hand}_{i}', f'y_hand_{hand}_{i}', f'z_hand_{hand}_{i}'),
        ]
        
        for x_col, y_col, z_col in patterns:
            if x_col in row.index:
                x = row[x_col]
                y = row[y_col]
                z = row[z_col]
                
                if pd.notna(x) and pd.notna(y) and pd.notna(z):
                    landmarks.append(Point3D(x=float(x), y=float(y), z=float(z)))
                break
    
    return landmarks if len(landmarks) == 21 else None

print("Landmark extraction function defined")

In [ ]:
# Extract and visualize sample landmarks
if 'df' in dir():
    # Try to extract landmarks from first valid row
    sample_landmarks = None
    
    for idx, row in df.head(50).iterrows():
        landmarks = extract_landmarks_from_row(row, 'right')
        if landmarks:
            sample_landmarks = landmarks
            print(f"Found valid landmarks at row {idx}")
            break
    
    if sample_landmarks:
        print(f"Extracted {len(sample_landmarks)} landmarks")
        plot_hand_3d(sample_landmarks, title="Sample Hand from Kaggle Data")
        plt.show()
    else:
        print("Could not extract landmarks. Check column naming patterns.")
        print("\nAvailable columns containing 'hand':")
        hand_cols = [c for c in df.columns if 'hand' in c.lower()]
        print(hand_cols[:20])

In [ ]:
# Plot all three views
if 'sample_landmarks' in dir() and sample_landmarks:
    fig = plot_hand_views(sample_landmarks, title="Sample Hand - All Views")
    plt.show()

In [ ]:
# Collect multiple samples for distribution analysis
all_samples = []

if 'df' in dir():
    for idx, row in df.iterrows():
        landmarks = extract_landmarks_from_row(row, 'right')
        if landmarks:
            all_samples.append(landmarks)
        
        if len(all_samples) >= 100:  # Limit for exploration
            break
    
    print(f"Collected {len(all_samples)} valid samples")

In [ ]:
# Plot coordinate distributions
if all_samples:
    fig = plot_landmark_distributions(
        all_samples,
        landmark_indices=[WRIST, THUMB_TIP, INDEX_FINGER_TIP, MIDDLE_FINGER_TIP, PINKY_TIP],
        figsize=(15, 12)
    )
    plt.show()

In [ ]:
# Variance analysis across samples
if all_samples and len(all_samples) > 5:
    fig = plot_variance_heatmap(all_samples, title="Landmark Variance Across Samples")
    plt.show()

## 3. Check for Data Quality Issues

In [ ]:
# Check for missing values in landmark columns
if 'df' in dir():
    hand_cols = [c for c in df.columns if 'hand' in c.lower()]
    
    if hand_cols:
        missing_counts = df[hand_cols].isnull().sum()
        missing_pct = (missing_counts / len(df) * 100).round(2)
        
        print("Missing value percentages:")
        print(missing_pct.describe())
        
        # Columns with high missing rates
        high_missing = missing_pct[missing_pct > 10]
        if len(high_missing) > 0:
            print(f"\nColumns with >10% missing: {len(high_missing)}")
            print(high_missing.sort_values(ascending=False).head(10))

In [ ]:
# Validate extracted landmarks
quality_results = []

if all_samples:
    for i, landmarks in enumerate(all_samples[:50]):
        result = validate_landmarks(landmarks)
        quality_results.append({
            'index': i,
            'is_valid': result.is_valid,
            'score': result.score,
            'errors': len(result.errors),
            'warnings': len(result.warnings)
        })
    
    quality_df = pd.DataFrame(quality_results)
    print("Validation Results:")
    display(quality_df.describe())
    
    valid_pct = quality_df['is_valid'].mean() * 100
    print(f"\nValid samples: {valid_pct:.1f}%")

In [ ]:
# Check for anatomical issues
issues_found = defaultdict(list)

if all_samples:
    for i, landmarks in enumerate(all_samples[:50]):
        x, y, z = landmarks_to_arrays(landmarks)
        
        # Check coordinate ranges
        if x.min() < -1 or x.max() > 2:
            issues_found['x_out_of_range'].append(i)
        if y.min() < -1 or y.max() > 2:
            issues_found['y_out_of_range'].append(i)
        if z.min() < -1 or z.max() > 1:
            issues_found['z_out_of_range'].append(i)
        
        # Check for collapsed hand (all points too close)
        spread = np.max([x.max() - x.min(), y.max() - y.min()])
        if spread < 0.05:
            issues_found['collapsed'].append(i)
        
        # Check for inverted hand (wrist below fingertips in Y)
        if landmarks[WRIST].y < landmarks[MIDDLE_FINGER_TIP].y:
            issues_found['possibly_inverted'].append(i)
    
    print("Issues found:")
    for issue, indices in issues_found.items():
        print(f"  {issue}: {len(indices)} samples")

In [ ]:
# Visualize problematic samples if any
if issues_found.get('collapsed'):
    print("Collapsed hand examples:")
    problematic = [all_samples[i] for i in issues_found['collapsed'][:3]]
    fig = plot_multiple_hands(problematic, titles=[f"Sample {i}" for i in issues_found['collapsed'][:3]])
    plt.show()

## 4. Compare Different Sign Samples

In [ ]:
# Group samples by sign label if available
samples_by_sign = defaultdict(list)

if 'df' in dir():
    # Find label column
    label_col = None
    for col in ['sign', 'label', 'sign_id', 'class']:
        if col in df.columns:
            label_col = col
            break
    
    if label_col:
        print(f"Using label column: {label_col}")
        
        for idx, row in df.head(500).iterrows():
            landmarks = extract_landmarks_from_row(row, 'right')
            if landmarks:
                sign = row[label_col]
                samples_by_sign[sign].append(landmarks)
        
        print(f"\nSigns with samples: {len(samples_by_sign)}")
        for sign, samples in list(samples_by_sign.items())[:10]:
            print(f"  {sign}: {len(samples)} samples")
    else:
        print("No label column found")

In [ ]:
# Compare samples of the same sign
if samples_by_sign:
    # Get a sign with multiple samples
    for sign, samples in samples_by_sign.items():
        if len(samples) >= 4:
            print(f"Comparing samples for sign: {sign}")
            fig = plot_multiple_hands(
                samples[:6],
                titles=[f"{sign} - Sample {i+1}" for i in range(6)],
                ncols=3,
                figsize=(15, 10)
            )
            plt.show()
            break

In [ ]:
# Compare different signs
if samples_by_sign and len(samples_by_sign) >= 4:
    different_signs = []
    sign_names = []
    
    for sign, samples in list(samples_by_sign.items())[:4]:
        if samples:
            different_signs.append(samples[0])
            sign_names.append(str(sign))
    
    if different_signs:
        print("Comparing different signs:")
        fig = plot_multiple_hands(different_signs, titles=sign_names, ncols=2)
        plt.show()

In [ ]:
# Analyze intra-sign variance
if samples_by_sign:
    variance_by_sign = {}
    
    for sign, samples in samples_by_sign.items():
        if len(samples) >= 3:
            # Stack all samples
            coords = []
            for s in samples:
                x, y, z = landmarks_to_arrays(s)
                coords.append(np.stack([x, y, z], axis=1))
            
            stacked = np.stack(coords, axis=0)
            variance = np.mean(np.var(stacked, axis=0))
            variance_by_sign[sign] = variance
    
    if variance_by_sign:
        var_df = pd.DataFrame({
            'sign': list(variance_by_sign.keys()),
            'variance': list(variance_by_sign.values())
        }).sort_values('variance', ascending=False)
        
        print("Intra-sign variance (higher = more variable poses):")
        display(var_df.head(10))
        
        plt.figure(figsize=(12, 4))
        plt.bar(range(len(var_df)), var_df['variance'])
        plt.xlabel('Sign')
        plt.ylabel('Mean Variance')
        plt.title('Pose Variance by Sign')
        plt.show()

## 5. Identify Preprocessing Needs

In [ ]:
# Analyze coordinate ranges
if all_samples:
    all_x, all_y, all_z = [], [], []
    
    for landmarks in all_samples:
        x, y, z = landmarks_to_arrays(landmarks)
        all_x.extend(x)
        all_y.extend(y)
        all_z.extend(z)
    
    print("Coordinate ranges:")
    print(f"  X: [{min(all_x):.4f}, {max(all_x):.4f}]")
    print(f"  Y: [{min(all_y):.4f}, {max(all_y):.4f}]")
    print(f"  Z: [{min(all_z):.4f}, {max(all_z):.4f}]")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(all_x, bins=50, alpha=0.7)
    axes[0].set_title('X Distribution')
    axes[0].set_xlabel('X')
    
    axes[1].hist(all_y, bins=50, alpha=0.7)
    axes[1].set_title('Y Distribution')
    axes[1].set_xlabel('Y')
    
    axes[2].hist(all_z, bins=50, alpha=0.7)
    axes[2].set_title('Z Distribution')
    axes[2].set_xlabel('Z')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Check hand sizes (scale normalization needed?)
if all_samples:
    hand_sizes = []
    
    for landmarks in all_samples:
        x, y, z = landmarks_to_arrays(landmarks)
        
        # Calculate bounding box size
        size = np.sqrt(
            (x.max() - x.min())**2 +
            (y.max() - y.min())**2 +
            (z.max() - z.min())**2
        )
        hand_sizes.append(size)
    
    print("Hand size statistics:")
    print(f"  Mean: {np.mean(hand_sizes):.4f}")
    print(f"  Std: {np.std(hand_sizes):.4f}")
    print(f"  Min: {np.min(hand_sizes):.4f}")
    print(f"  Max: {np.max(hand_sizes):.4f}")
    print(f"  Ratio (max/min): {np.max(hand_sizes)/np.min(hand_sizes):.2f}x")
    
    plt.figure(figsize=(10, 4))
    plt.hist(hand_sizes, bins=30, alpha=0.7)
    plt.axvline(np.mean(hand_sizes), color='red', linestyle='--', label='Mean')
    plt.xlabel('Hand Size (bounding diagonal)')
    plt.ylabel('Frequency')
    plt.title('Hand Size Distribution')
    plt.legend()
    plt.show()
    
    if np.max(hand_sizes)/np.min(hand_sizes) > 2:
        print("\n⚠️ Recommendation: Scale normalization is NEEDED (>2x size variation)")
    else:
        print("\n✓ Scale variation is acceptable")

In [ ]:
# Check wrist positions (translation normalization needed?)
if all_samples:
    wrist_positions = []
    
    for landmarks in all_samples:
        wrist = landmarks[WRIST]
        wrist_positions.append([wrist.x, wrist.y, wrist.z])
    
    wrist_arr = np.array(wrist_positions)
    
    print("Wrist position statistics:")
    print(f"  X - Mean: {wrist_arr[:, 0].mean():.4f}, Std: {wrist_arr[:, 0].std():.4f}")
    print(f"  Y - Mean: {wrist_arr[:, 1].mean():.4f}, Std: {wrist_arr[:, 1].std():.4f}")
    print(f"  Z - Mean: {wrist_arr[:, 2].mean():.4f}, Std: {wrist_arr[:, 2].std():.4f}")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(wrist_arr[:, 0], bins=30, alpha=0.7)
    axes[0].set_title('Wrist X Position')
    
    axes[1].hist(wrist_arr[:, 1], bins=30, alpha=0.7)
    axes[1].set_title('Wrist Y Position')
    
    axes[2].hist(wrist_arr[:, 2], bins=30, alpha=0.7)
    axes[2].set_title('Wrist Z Position')
    
    plt.tight_layout()
    plt.show()
    
    if wrist_arr[:, 0].std() > 0.1 or wrist_arr[:, 1].std() > 0.1:
        print("\n⚠️ Recommendation: Translation normalization is NEEDED")
    else:
        print("\n✓ Position variation is acceptable")

In [ ]:
# Summary of preprocessing recommendations
print("="*60)
print("PREPROCESSING RECOMMENDATIONS")
print("="*60)

recommendations = []

if 'hand_sizes' in dir():
    if np.max(hand_sizes)/np.min(hand_sizes) > 1.5:
        recommendations.append("1. Scale normalization - hands vary significantly in size")

if 'wrist_arr' in dir():
    if wrist_arr[:, 0].std() > 0.05 or wrist_arr[:, 1].std() > 0.05:
        recommendations.append("2. Translation normalization - wrist positions vary")

if issues_found.get('collapsed'):
    recommendations.append(f"3. Filter collapsed hands - {len(issues_found['collapsed'])} found")

if issues_found.get('possibly_inverted'):
    recommendations.append(f"4. Check/fix inverted hands - {len(issues_found['possibly_inverted'])} found")

if 'quality_df' in dir():
    invalid_pct = (1 - quality_df['is_valid'].mean()) * 100
    if invalid_pct > 5:
        recommendations.append(f"5. Quality filtering - {invalid_pct:.1f}% samples have issues")

recommendations.append("6. Consider rotation normalization for consistent orientation")

for rec in recommendations:
    print(f"  {rec}")

print("\n" + "="*60)

## Next Steps

Based on this analysis, proceed to:
1. **02_normalization_experiments.ipynb** - Test normalization approaches
2. **03_tolerance_analysis.ipynb** - Analyze variance for tolerance calculation
3. **04_angle_calculations.ipynb** - Verify angle calculations
4. **05_quality_metrics.ipynb** - Develop quality scoring